# Yelp GNN — full data + FOCAL LOSS, *properly tuned* (Colab GPU)

The first attempt used inverse-frequency alpha (~7x toward fake) — far more aggressive than standard focal (~0.25) — which over-predicted (40%) and squashed the probabilities. **That was a mis-config, not a verdict on focal loss.**

This version **sweeps the rebalancing strength** and reports, for each, the **predicted-fake rate** (calibration — we want it near the true ~12%) alongside macro-F1. It trains on **all ~1.03M reviews** (`graph_full.npz`) and saves every config's predictions so we can compare ring extraction offline.

**Setup:** GPU runtime. Upload `data/yelp_graph/graph_full.npz`. Run all → `yelp_gnn_focal_preds.npz`.

In [ ]:
!pip -q install torch_geometric
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
import numpy as np, torch
from torch_geometric.data import HeteroData
GRAPH = 'graph_full.npz'
d = np.load(GRAPH)
def zscore(a):
    a = torch.tensor(a, dtype=torch.float); m, s = a.mean(0, keepdim=True), a.std(0, keepdim=True)
    return (a - m) / (s + 1e-6)
data = HeteroData()
data['review'].x   = zscore(d['review_feat'])
data['reviewer'].x = zscore(d['reviewer_feat'])
data['business'].x = zscore(d['business_feat'])
rid = torch.arange(d['review_feat'].shape[0])
ru  = torch.tensor(d['review_user'], dtype=torch.long); rb = torch.tensor(d['review_biz'], dtype=torch.long)
data['review','by','reviewer'].edge_index    = torch.stack([rid, ru])
data['reviewer','wrote','review'].edge_index  = torch.stack([ru, rid])
data['review','on','business'].edge_index     = torch.stack([rid, rb])
data['business','has','review'].edge_index     = torch.stack([rb, rid])
data['review'].y = torch.tensor(d['review_label'], dtype=torch.long)
split = torch.tensor(d['review_split'], dtype=torch.long)
data['review'].train_mask = (split==1); data['review'].val_mask=(split==2); data['review'].test_mask=(split==3)
print(data); print('fake rate', float((data['review'].y==1).float().mean()))

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv, Linear
REL = [('review','by','reviewer'),('reviewer','wrote','review'),('review','on','business'),('business','has','review')]
class HeteroGNN(torch.nn.Module):
    def __init__(self, hidden=128, layers=2, out=2):
        super().__init__()
        self.proj = torch.nn.ModuleDict({nt: Linear(-1, hidden) for nt in ['review','reviewer','business']})
        self.convs = torch.nn.ModuleList([HeteroConv({r: SAGEConv((-1,-1), hidden) for r in REL}, aggr='sum') for _ in range(layers)])
        self.dropout = torch.nn.Dropout(0.3); self.out = Linear(hidden, out)
    def forward(self, x, e):
        x = {k: self.proj[k](v).relu() for k,v in x.items()}
        for c in self.convs: x = {k: self.dropout(v.relu()) for k,v in c(x,e).items()}
        return self.out(x['review'])
def focal_loss(logits, t, gamma, alpha):
    logp = F.log_softmax(logits,1); logpt = logp.gather(1,t.unsqueeze(1)).squeeze(1); pt = logpt.exp()
    loss = -((1-pt)**gamma)*logpt
    if alpha is not None: loss = alpha.gather(0,t)*loss
    return loss.mean()

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); data = data.to(dev)
tr,va,te = data['review'].train_mask, data['review'].val_mask, data['review'].test_mask
yb = data['review'].y; ytr = yb[tr].cpu().numpy(); n0=(ytr==0).sum(); n1=(ytr==1).sum()
def A(w0,w1): return torch.tensor([w0,w1], dtype=torch.float, device=dev)
# sweep the rebalancing strength: from none -> mild -> strong -> (the original over-aggressive)
CONFIGS = [
  ('CE  no-weight',      'ce',    None, None),
  ('Focal g2 no-alpha',  'focal', 2.0,  None),
  ('Focal g2 a=[1,2]',   'focal', 2.0,  A(1.0,2.0)),
  ('Focal g2 a=[1,4]',   'focal', 2.0,  A(1.0,4.0)),
  ('Focal g1 a=[1,2]',   'focal', 1.0,  A(1.0,2.0)),
  ('CE invfreq (orig)',  'ce',    None, A(len(ytr)/(2*n0), len(ytr)/(2*n1))),
]
def evaluate(model, mask):
    model.eval()
    with torch.no_grad(): pr = model(data.x_dict, data.edge_index_dict).argmax(1)
    yt=yb[mask].cpu().numpy(); yp=pr[mask].cpu().numpy()
    return f1_score(yt,yp,average='macro',zero_division=0), recall_score(yt,yp,pos_label=1,zero_division=0), precision_score(yt,yp,pos_label=1,zero_division=0)
def run(kind, gamma, alpha):
    torch.manual_seed(0)
    model = HeteroGNN().to(dev)
    with torch.no_grad(): model(data.x_dict, data.edge_index_dict)
    opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=5e-4)
    best, state, bad = -1, None, 0
    for ep in range(1,151):
        model.train(); opt.zero_grad(); out = model(data.x_dict, data.edge_index_dict)
        loss = F.cross_entropy(out[tr], yb[tr], weight=alpha) if kind=='ce' else focal_loss(out[tr], yb[tr], gamma, alpha)
        loss.backward(); opt.step()
        v = evaluate(model, va)[0]
        if v>best: best, state, bad = v, {k:vv.cpu().clone() for k,vv in model.state_dict().items()}, 0
        else: bad += 1
        if bad>=15: break
    model.load_state_dict(state)
    with torch.no_grad(): prob = torch.softmax(model(data.x_dict, data.edge_index_dict),1)[:,1].cpu().numpy()
    mf1,fr,fp = evaluate(model, te)
    return prob, mf1, fr, fp
saved = {'review_user': d['review_user'], 'review_biz': d['review_biz'], 'label': d['review_label'], 'split': d['review_split']}
print(f"{'config':20}{'testMF1':>9}{'fakeR':>7}{'fakeP':>7}{'pred-fake%':>11}  (true 12.2%)")
for i,(name,kind,gamma,alpha) in enumerate(CONFIGS):
    prob, mf1, fr, fp = run(kind, gamma, alpha)
    saved[f'prob_{i}'] = prob
    print(f"{name:20}{mf1:>8.1%}{fr:>7.1%}{fp:>7.1%}{float((prob>=0.5).mean()):>10.1%}")
saved['config_names'] = np.array([c[0] for c in CONFIGS])
np.savez_compressed('yelp_gnn_focal_preds.npz', **saved)
print('\nsaved yelp_gnn_focal_preds.npz (prob_0..prob_%d + structure + labels)' % (len(CONFIGS)-1))